# L12b: Why do Markets Crash?
In this lecture, we study how a small change in the rule that individual traders follow can turn a calm market into a falling one. We build a simple agent-based market in which traders on a grid act on the intentions of their neighbors and trade through a dealer, and we change a single entry of the rule that some of the traders follow.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Understand a trading rule as a policy:__ Recognize that a trader's rule is a lookup table from what its neighbors intend to what it intends next, and explain how changing one entry of the table changes the trader's behavior in one kind of neighborhood only.
> * __Understand how a dealer links trades to prices:__ Explain how net buying from the dealer raises the price and net selling lowers it, why intentions can spread among traders who cannot complete a trade, and why a switched market must be compared with a control that shares its history.
> * __Interpret a rule-switch simulation:__ Read the fraction of traders who intend to sell and the dealer's price for a switched market and its control, and explain what the comparison does and does not tell us about real market crashes.

Some results in this lecture are core, and some are here to build understanding:

> __What you need to know:__
>
> __Core concepts:__
> * A Wolfram rule is a policy. The state is the neighbor sum $c\in\{0,1,\dots,8\}$, the sum of the intentions of a trader's four neighbors (Buy $=0$, Hold $=1$, Sell $=2$), and the action is the trader's next intention. The rule number writes the nine actions in base 3, and Rules 19280 and 19286 differ in one entry, at $c=1$: Buy versus Sell.
> * A dealer turns trades into a price: net buying from the dealer raises the price, and net selling lowers it. Buy and Sell are public intentions that neighbors read without checking holdings, and a shallow dealer can crash the price without any switch, so we compare every switched market with a control.
> * Giving the changed rule to a fraction $f$ of the traders after a shared history can tip the market toward a Sell majority while the control does not. This is an analogy for how a change in behavior can spread through a market; it does not show that such a change causes real crashes.
>
> __For understanding__ (you will not need to reproduce or memorize these):
> * The price update formula of the dealer, and the contrarian band rule of the technical traders.
> * The held, active, and random-signal conditions for the traders on the edge of the grid.
> * The Q-learning material in the optional advanced notebook.

__Notation:__ The [course notation page](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/faq/notation.html#L12b) lists this lecture's symbols with their meanings and units.

Let's get started!
___

## Examples
__We will work through this example in lecture:__

> [▶ Switch one entry of a Wolfram market rule](CHEME-5660-L12b-Example-Wolfram-RuleSwitch-Fall-2026.ipynb). In this example, we print the lookup tables of two Wolfram rules and find the one neighbor sum at which they differ. We plot the fraction of traders who intend to sell and the dealer's price for a market in which some traders switch rules, beside a control that shares its history and never switches. We repeat the comparison with edge traders who react to the price and edge traders who do not, and with a deep and a shallow dealer. We then count how often a Sell majority forms across many starting markets, and run a small switched market ourselves, changing the fraction of traders who switch, the dealer's depth, and the edge condition.

Optional advanced notebooks that extend today's material are listed in the Optional Advanced Material section at the end of this lecture.
___

<div>
    <center>
        <img src="figs/Fig-Crashes-Schematic-2024.png" width="880"/>
    </center>
</div>

## Why do markets crash?
Market crashes are sudden and significant declines in asset prices across a broad market. These are sometimes referred to as __black swan__ events due to their unexpected nature and severe impact. Understanding the causes of market crashes is crucial for investors, policymakers, and economists. Here are three landmark episodes that show how investor psychology swings from euphoria to panic:

> __Historical Market Crashes__
>
> - **1929 Wall Street Crash (Oct 1929)**: Margin-fueled speculation and slowing earnings met a sudden loss of confidence; euphoria flipped to fear as leveraged investors rushed to cover. The crash fed bank failures and a deep slump, prompting reforms that later created the SEC and stricter disclosure rules. [Federal Reserve History](https://www.federalreservehistory.org/essays/stock-market-crash-of-1929) · [FRASER archival docs](https://fraser.stlouisfed.org/title/stock-market-1929-revisited-3599)  · [The lessons of the Stock Market crash of 1929 - CBS News](https://varnerlab.org/watch.html?v=26WMF7Xdb0E).
> - **1987 Black Monday (Oct 19, 1987)**: Portfolio insurance and program trading amplified a slide; herd-like selling fed on itself once screens turned red. The one-day 22.6% Dow drop led to circuit breakers and cross-market coordination. [Presidential Task Force on Market Mechanisms, 1988 (Brady Report)](https://fraser.stlouisfed.org/title/report-presidential-task-force-market-mechanisms-5732)
> - **April 2025 Tariff Mini-Crash**: Tariff announcements and retaliation fears triggered fast algorithmic selling as investors reassessed earnings and supply chains; policy uncertainty hit sentiment first, fundamentals second. Losses eased after policy clarifications, but the episode underscored how policy shocks and crowding can spark sharp swings. [USTR press releases, April 2025](https://ustr.gov/about-us/policy-offices/press-office/press-releases/2025/april)
>
> The unifying theme across these crashes is the interplay of investor psychology, market dynamics, and external shocks. Euphoria and greed can drive prices to unsustainable levels, while fear and panic can lead to rapid sell-offs. Understanding these dynamics is crucial for managing risk and making informed investment decisions.

Markets are man-made systems, and like all man-made systems, they can fail. One (bottom up) approach to understanding market crashes is agent-based modeling.

## Agent-Based Modeling of Market Crashes
Agent-based models (ABMs) simulate financial markets by representing individual traders as autonomous agents who follow decision rules and interact with each other. Each agent has its own strategy, information, and objectives. The market price emerges from the collective actions of these agents rather than from a single equilibrium equation. Previously, we studied two classical agent-based models, [the minority game](https://arxiv.org/abs/adap-org/9708006) and the [Farmer–Joshi market](https://arxiv.org/abs/cond-mat/0012419). Two others show how interaction among traders can produce large moves:

> __Other Agent-Based Financial Models__
>
> - **Iori Microsimulation Model (2002)**: Heterogeneous interacting traders with communication and imitation generate synchronization effects and large fluctuations in returns. A trade friction mechanism responds to price movements and creates feedback that generates volatility clustering. [arXiv:adap-org/9905005](https://arxiv.org/abs/adap-org/9905005)
> - **Lux-Marchesi Model (1999)**: Agents switch between fundamentalist and chartist strategies based on profitability. The model generates bubbles and crashes through herding behavior and strategy imitation. [Nature 397, 498-500 (1999)](https://doi.org/10.1038/17290)

Today, we use a market in which traders sit on a grid and act on what their neighbors intend to do. Our question is the one in the title: can a small change in the psychology that drives individual traders' actions produce a crash?

## Agents as Policies
To simulate a market of agents, we need a rule that tells each agent what to do given what it sees. We call this rule a __policy__: a table that gives, for each possible state of the agent's surroundings, the action the agent takes. In our market, the state is a number that summarizes what a trader's four neighbors intend to do (defined below), and the action is the trader's own next intention: Buy, Hold, or Sell.

Where could such a table come from? __Q-learning__ is one answer. It is a model-free reinforcement learning method with which an agent learns the table from experience, by estimating how good each action is in each state and then choosing the best one. Our traders are not learning: they are handed their tables, and we study what happens when we change one of them. The optional advanced notebook at the end of this lecture develops Q-learning in full.

## Wolfram Markets
Wolfram markets are agent-based models of financial markets in which simple rules and local interactions among agents produce market-level behavior. The idea is inspired by Stephen Wolfram's work on cellular automata and complex systems. The version we use follows [our paper on rule switches in Wolfram markets](https://github.com/varnerlab/Wolfram-markets-abm-paper).

> __Wolfram Markets Overview__
>
> Agents are traders placed on the cells of a square grid. Each trader holds one of three __intentions__: Buy, Hold, or Sell. We code them $0$, $1$, and $2$ and write $\iota_{j}(t)\in\{0,1,2\}$ for the intention of trader $j$ at step $t$. Each trader on the interior of the grid has four neighbors, above, below, left, and right, and updates its intention from theirs. This is a three-color two-dimensional cellular automaton: [Chapter 3, A World of Simple Programs, A New Kind of Science, 2002](https://cornell.box.com/s/10md9xuf184a6s3c25n6005uuia7da1e)

Let $\partial i$ denote the four neighbors of trader $i$. The trader looks only at the __neighbor sum__ $c_{i}(t)$, the sum of its neighbors' intentions, and not at its own current intention:
$$
\underbrace{c_{i}(t)}_{\text{neighbor sum}} = \sum_{j\in\partial i}\iota_{j}(t)\in\{0,1,\dots,8\}
$$
A neighbor sum of $0$ means four neighbors intend to Buy, and a sum of $8$ means four intend to Sell. The policy is a table with one action $a_{c}\in\{0,1,2\}$ for each of the nine possible sums $c=0,1,\dots,8$: the intention the trader adopts next when its neighbor sum is $c$. There are $3^{9}=19{,}683$ such tables, and we identify each by its __rule number__, the table written in base 3:
$$
\underbrace{\sum_{c=0}^{8}a_{c}\,3^{c}}_{\text{rule number}}
$$
The digits are written from $a_{8}$ on the left to $a_{0}$ on the right. We use Rule 19280, $(222110002)_{3}$, as the baseline.

Traders on the edge of the grid are different. Some of them are __technical traders__ who form their intentions from the dealer's price, and the others stay in Hold. A technical trader follows a contrarian band rule: it compares the price with a band around a recent average of the price, intends to Buy when the price falls below the band, to Sell when it rises above it, and to Hold otherwise. Interior traders next to the edge include these intentions in their neighbor sums.

Each step of the simulation then runs in this order:
1. __Rule changes and orders:__ Any scheduled rule change is applied. Each trader submits an order from its current intention: a Buy requests one share, a Sell offers one share, and a Hold submits no order.
2. __Clearing:__ The dealer matches the orders, completes the trades it can, and updates the price.
3. __Technical traders:__ Each technical trader updates its band and its intention from the new price.
4. __Interior traders:__ All interior traders update together, using the previous intentions of their interior neighbors and the new intentions of the technical traders next to them. These intentions become the orders of the next step.

## The Rule Switch
Rule 19286, $(222110022)_{3}$, differs from the baseline in one entry. When three neighbors intend to Buy and one intends to Hold, the neighbor sum is $c=1$. A trader following Rule 19280 then intends to Buy, and a trader following Rule 19286 intends to Sell. All the other eight entries are the same. The figure shows the neighborhood in panel A and the two lookup tables in panel B. The base-3 digits of a rule number run from $c=8$ down to $c=0$, while the columns of the table in panel B run from $c=0$ to $c=8$.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L12b-RuleSwitch.svg" width="780" alt="Panel A shows a trader whose four neighbors have intentions 0, 0, 0, and 1 (three Buy, one Hold), so the neighbor sum is c = 0 + 0 + 0 + 1 = 1. An arrow leads to Buy under Rule 19280 and another arrow leads to Sell under Rule 19286. Panel B lists the nine entries of the two lookup tables for neighbor sums c = 0 to 8. Rule 19280 reads 2, 0, 0, 0, 1, 1, 2, 2, 2 and Rule 19286 reads 2, 2, 0, 0, 1, 1, 2, 2, 2, so the tables differ only in the column c = 1, which a dashed box marks. Buy is blue, Hold is gray, and Sell is orange."/>
    </center>
</div>

We use this one-entry difference as a model of a change in psychology: some traders start to respond to the same neighborhood with selling instead of buying. The experiment has three parts:
* __Shared history:__ All interior traders follow Rule 19280 while the market develops. At the switch step, a fraction $f$ of the interior traders, chosen in advance, receive Rule 19286. They keep their current intentions, cash, and shares, and use the new rule to choose their next intentions.
* __Control:__ A second market has exactly the same history and never switches. Any difference between the two markets after the switch step is caused by the rule change.
* __Edge traders:__ In the __active__ condition, the technical traders react to the dealer's price. In the __held__ condition, we keep them in Hold, so price changes cannot reach the interior intentions and any cascade must come from traders reading their neighbors. The example also has a random-signal condition, in which each technical trader draws a new intention at random at every step; this is a course addition, not a result from the paper.

Let's see the experiment run:

> __Example__
>
> [▶ Switch one entry of a Wolfram market rule](CHEME-5660-L12b-Example-Wolfram-RuleSwitch-Fall-2026.ipynb). In this example, we plot the fraction of traders who intend to sell and the dealer's price for a switched market beside its control, count how often a Sell majority forms across many starting markets, and run a small switched market ourselves.

The example uses the dealer's price throughout, so let's look at how the dealer sets it.

## Prices and the Dealer
The dealer holds cash and shares and takes the other side of the trades that the traders do not match with each other. Three points are enough to read the results of the example.

__1. The dealer sets the price from the net trade.__ Let $x_{t}$ be the net number of shares the dealer sells to the traders at step $t$, positive when traders are net buyers and negative when they are net sellers. Let $Q_{t-1}$ be the number of shares the dealer holds before the trade, and $P_{t-1}$ the price before the trade. The price after the trade is
$$
P_{t} = P_{t-1}\Bigl(\underbrace{\frac{Q_{t-1}}{Q_{t-1}-x_{t}}}_{\substack{\text{shares before}\\\text{over shares after}}}\Bigr)^{2}
$$
When traders are net buyers, $x_{t}>0$, the dealer is left with fewer shares, the ratio exceeds one, and the price rises. When traders are net sellers, $x_{t}<0$, the ratio is below one and the price falls. The dealer's depth $h$, its initial number of shares per trader, sets the size of $Q_{t-1}$: the same net order is a smaller fraction of a deep dealer's shares and moves the price less.

__2. Buy and Sell are public intentions.__ Neighbors read a trader's intention without checking whether the trader holds the cash or the shares to act on it. An intention can therefore spread through the grid even when its sender cannot complete a trade. A completed trade does need resources: a buyer needs cash to pay the price, and a seller needs a share.

__3. Dealer depth matters, so we use a control.__ A shallow dealer can crash the price with no rule switch at all, because modest net selling is already a large fraction of its shares. We therefore never read a fall in the price of a switched market on its own, and always compare it with the no-switch control that shares its history.

The full clearing rule, which fixes the direction and size of the net trade and allocates the orders, is in the example's code and in the paper. The paper also studies trader resources, transaction fees, and a variant in which traders see each other's holdings; we leave those results to the paper.

## Back to Real Crashes
Our market is a toy, and we do not claim that it explains any real crash. The rule switch is an analogy, and each of the three episodes above can be read in its terms in a loose way:
* __1929:__ Rampant speculation drew in traders who bought because others were buying, and some people worry about speculation again today. In our market, that is a population whose rule turns neighbors' Buy intentions into Buy intentions, which leaves the market sensitive to a change in the rule of some traders.
* __1987:__ Portfolio insurance told a subset of investors to sell as prices fell. In our market, that is a fraction $f$ of the traders receiving a different entry in their rule, which changed how a part of the market responded to the same conditions.
* __April 2025:__ News about tariffs changed how traders read each other. In our market, that is a change in what the same neighborhood of intentions means to a trader, and therefore in what the trader does next.

## Optional Advanced Material
> [▶ Learn a policy with Q-learning](advanced/qlearning/CHEME-5660-L12b-Advanced-QLearning-Fall-2026.ipynb). In this notebook, we ask how a trader could learn a policy table from experience instead of being handed one. We write the Q-learning update for the value of each action in each state, lay out the algorithm step by step, and describe the conditions under which it converges to the best policy.
___

## Summary
In this lecture, we built a simple agent-based market and used it to ask how a small change in trader behavior can tip a market toward selling.

> __Key Takeaways:__
>
> * __A trading rule is a policy, and one entry of it can change:__ We read a Wolfram rule as a lookup table from the sum of a trader's four neighbors' intentions to the trader's next intention, and wrote the table as a rule number in base 3. We compared two rules that differ only when three neighbors intend to buy and one intends to hold, where one rule buys and the other sells. This let us describe a change in the psychology of a trader as the change of a single entry.
> * __A dealer turns trades into a price, and a control isolates the switch:__ We used the dealer's price update to connect net buying and net selling to the price, and saw that intentions are public, so neighbors react to them even when a trade cannot be completed. We compared every switched market with a control that shares its history and never switches. This let us attribute the differences after the switch to the rule change, and showed that a shallow dealer can move the price a long way even without one.
> * __A switch for some traders can tip the market, but this is an analogy:__ We gave the changed rule to a fraction of the traders after a shared history, and followed the fraction of traders who intend to sell and the price with edge traders who react to the price and edge traders who do not. This showed that the market can turn toward a Sell majority, and we related that to three real crashes as an analogy. A simulation does not show that heterogeneity alone causes real crashes; update order, network structure, boundary rules, calibration, and common shocks also affect the result.

Agent-based models give us a way to ask how the rules of individual traders add up to the behavior of a market.
___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___